# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fiona-ali/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

**Lane:** Refresh / Content Opportunity Scoring. **Decision:** which pages should an editor inspect first for a possible refresh?

This run uses the repo's 30,000-row anonymized starter CSV (one row per content item; 44 columns). It creates a transparent rule baseline and a ranked review queue. The `trend_direction == "down"` field is only an observed, trailing-window proxy used for signal checks and descriptive Precision@K; it is never used to score or rank a page. This starter file does not provide an independent future outcome window, so the evaluation is a descriptive baseline check, not a forward-looking estimate or a causal claim.

## 1. Rule, signals, and reason code

**Rule in plain words:** among pages with at least 100 impressions in the supplied 90-day snapshot, rank pages higher when they combine more search visibility with more time since the last update. Cap the age factor at 180 days so older pages do not gain unlimited points.

**Score:** `log1p(impressions_90d) × min(days_since_last_update, 180) / 180`. The logarithm reduces the effect of extreme traffic counts; the capped age factor is between 0 and 1. The score is a review priority, not a probability of decline.

**Single reason code:** `visibility_and_staleness`. **Action label:** `editor_review`. The label means “inspect before deciding”; it does not instruct an automatic edit or publish.

The signal checks use (1) update age, linked to the session's real refresh-flag logic, and (2) impression volume, a visibility signal. Bucket tables print the row count `n` and the observed decline-proxy rate. Verdicts are deliberately cautious because buckets are not a causal test.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")
OUT_PATH = Path("work/outputs/baseline_action_score.csv")
METRICS_PATH = Path("work/outputs/w04_baseline_metrics.json")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Expected data/raw/content_refresh_anonymized.csv from the repository starter data. "
        "Do not replace it with warehouse data or a June sample for this baseline."
    )

df = pd.read_csv(DATA_PATH)
required = {"content_id", "client_id", "impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction"}
missing = sorted(required - set(df.columns))
if missing:
    raise ValueError(f"Required starter-data columns missing: {missing}")
if df.empty:
    raise ValueError("Starter data is empty")

# Target proxy is isolated for evaluation and signal descriptions; never pass it to the score.
df["decline_proxy"] = df["trend_direction"].astype("string").str.lower().eq("down").astype(int)
print(f"Loaded {len(df):,} rows × {len(df.columns)-1:,} source columns (plus one local evaluation field).")
print(f"Observed decline-proxy base rate in the full starter slice: {df['decline_proxy'].mean():.3f}")
print("The evaluation label is descriptive and trailing-window; it is not a future outcome.")

Loaded 30,000 rows × 44 source columns (plus one local evaluation field).
Observed decline-proxy base rate in the full starter slice: 0.542
The evaluation label is descriptive and trailing-window; it is not a future outcome.


### Signal 1 — days since last update (refresh-flag-linked)

**Verdict: MIXED.** The 91–180-day bucket has a higher decline-proxy rate than 0–90 days, but the 181–365-day bucket is lower and contains only 169 rows. The pattern is not monotonic, so age is a cautious review cue, not proof that stale pages will decline. The oldest bucket is shown separately rather than hidden.

In [2]:
freshness_bins = [-1, 90, 180, 365, np.inf]
freshness_labels = ["0-90", "91-180", "181-365", "366+"]
fresh = df.copy()
fresh["freshness_bucket"] = pd.cut(
    pd.to_numeric(fresh["days_since_last_update"], errors="coerce"),
    bins=freshness_bins, labels=freshness_labels
)
fresh_table = (
    fresh.groupby("freshness_bucket", observed=False)
    .agg(n=("content_id", "size"), decline_proxy_rate=("decline_proxy", "mean"),
         median_impressions=("impressions_90d", "median"))
    .reset_index()
)
fresh_table["decline_proxy_rate"] = fresh_table["decline_proxy_rate"].round(3)
display(fresh_table)

freshness_bucket     n  decline_proxy_rate  median_impressions
            0-90 20655               0.512               472.0
          91-180  9171               0.611              1692.0
         181-365   169               0.467                16.0
            366+     5               0.600                 2.0


### Signal 2 — 90-day impression volume

**Verdict: MIXED.** Middle-volume buckets have higher decline-proxy rates than the lowest and highest buckets. This is not a monotonic relationship. Volume is used to represent visibility and editorial reach in the priority score; it is not treated as evidence that high-volume pages are more likely to decline.

In [3]:
volume_bins = [-np.inf, 99, 499, 2999, 9999, np.inf]
volume_labels = ["<100", "100-499", "500-2,999", "3,000-9,999", "10,000+"]
volume = df.copy()
volume["volume_bucket"] = pd.cut(
    pd.to_numeric(volume["impressions_90d"], errors="coerce"),
    bins=volume_bins, labels=volume_labels
)
volume_table = (
    volume.groupby("volume_bucket", observed=False)
    .agg(n=("content_id", "size"), decline_proxy_rate=("decline_proxy", "mean"),
         median_impressions=("impressions_90d", "median"))
    .reset_index()
)
volume_table["decline_proxy_rate"] = volume_table["decline_proxy_rate"].round(3)
display(volume_table)

volume_bucket    n  decline_proxy_rate  median_impressions
         <100 7994               0.389                12.0
      100-499 5280               0.604               251.5
    500-2,999 8443               0.621              1230.0
  3,000-9,999 4681               0.606              5237.0
      10,000+ 3602               0.524             20063.0


## 2. Build the ranked queue (writes the CSV)

The eligibility floor is 100 impressions. For eligible pages only, the rule scores `log1p(impressions_90d) × min(days_since_last_update, 180) / 180`. The score uses only impression volume and update age. IDs are included in the queue for grouping and follow-up, never as score inputs. Each row receives exactly one reason code and one action label.

In [4]:
eligible = df.loc[pd.to_numeric(df["impressions_90d"], errors="coerce") >= 100].copy()
eligible["score"] = (
    np.log1p(pd.to_numeric(eligible["impressions_90d"], errors="coerce").clip(lower=0))
    * pd.to_numeric(eligible["days_since_last_update"], errors="coerce").clip(lower=0, upper=180)
    / 180.0
)
eligible["reason_code"] = "visibility_and_staleness"
eligible["action_label"] = "editor_review"

# Stable tie-breaking makes reruns reproducible and does not use IDs as features.
queue = eligible.sort_values(
    ["score", "impressions_90d", "days_since_last_update"],
    ascending=[False, False, False], kind="mergesort"
).reset_index(drop=True)
queue.insert(0, "rank", np.arange(1, len(queue) + 1))
queue["score"] = queue["score"].round(6)

queue_columns = [
    "rank", "content_id", "client_id", "score", "reason_code", "action_label",
    "impressions_90d", "days_since_last_update", "avg_position", "ctr"
]
queue = queue[queue_columns]
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
queue.to_csv(OUT_PATH, index=False)
print(f"Eligible pages: {len(queue):,} / {len(df):,} (impressions_90d >= 100)")
print(f"Wrote ranked queue: {OUT_PATH}")
print("Score input columns: impressions_90d, days_since_last_update")
print("Excluded from score: trend_direction, trend_pct, IDs, and all label-derived fields")
assert len(queue) == int((df["impressions_90d"] >= 100).sum())
assert queue["reason_code"].eq("visibility_and_staleness").all()
assert queue["action_label"].eq("editor_review").all()

Eligible pages: 22,006 / 30,000 (impressions_90d >= 100)
Wrote ranked queue: work/outputs/baseline_action_score.csv
Score input columns: impressions_90d, days_since_last_update
Excluded from score: trend_direction, trend_pct, IDs, and all label-derived fields


## 3. Top-10 review

Each row includes its action, why the rule ranked it, and a concrete reason the recommendation could be wrong. A high score is only a prompt for human review.

In [5]:
top10 = queue.head(10).copy()

def why_here(row):
    return (f"{int(row.impressions_90d):,} impressions and {int(row.days_since_last_update)} days since update; "
            f"score={row.score:.3f} under the stated visibility × capped-age rule.")

def could_be_wrong(row):
    if row.days_since_last_update < 180:
        return (f"Updated {int(row.days_since_last_update)} days ago (below the 180-day cap); "
                "the high impression count may dominate despite limited staleness.")
    if pd.notna(row.avg_position) and row.avg_position > 0 and row.avg_position <= 10 and row.ctr >= 1:
        return (f"Already averages position {row.avg_position:g} with CTR {row.ctr:g}%; "
                "it may already work well, so a refresh could disrupt useful content.")
    if pd.isna(row.avg_position) or row.avg_position == 0:
        return "Position is missing/no-data; without a usable rank, the review priority lacks ranking context."
    return (f"The rule cannot distinguish outdated content from a demand shift; position {row.avg_position:g} "
            "alone does not establish that an edit will help.")

top10["why_ranked"] = top10.apply(why_here, axis=1)
top10["what_could_make_it_wrong"] = top10.apply(could_be_wrong, axis=1)
top10["action"] = "editor_review"
review = top10[["rank", "content_id", "action", "reason_code", "why_ranked", "what_could_make_it_wrong"]]
display(review)
assert len(review) == 10

 rank           content_id        action              reason_code                                                                                                why_ranked                                                                                                         what_could_make_it_wrong
    1 content_cf56e2e2e282 editor_review visibility_and_staleness 61,678 impressions and 194 days since update; score=11.030 under the stated visibility × capped-age rule. The rule cannot distinguish outdated content from a demand shift; position 19.7 alone does not establish that an edit will help.
    2 content_7368877ea310 editor_review visibility_and_staleness 59,472 impressions and 194 days since update; score=10.993 under the stated visibility × capped-age rule. The rule cannot distinguish outdated content from a demand shift; position 24.8 alone does not establish that an edit will help.
    3 content_1bfaa38ff26c editor_review visibility_and_staleness 25,715 impressions and 194 days

## 4. Weak picks, evaluation, and leakage check

**Weak-pick note:** rank 10 is a useful counterexample: it is only 151 days since update, below the 180-day age cap, and its CTR is 2.45%. Its high impression count lifts its score, but those facts alone do not show that its content needs work. This is exactly why the action is editor review rather than an automatic refresh.

The provided decline proxy is evaluated only after ranking. Precision@K here is a descriptive check against the starter CSV's trailing-window trend label; it is not an independent future test. It must be read next to the eligible-set base rate. The rule is intentionally simple and is the frozen reference for later model comparison.

In [6]:
eligible_label = eligible["decline_proxy"].to_numpy(dtype=int)
ranked_label = queue["content_id"].map(eligible.set_index("content_id")["decline_proxy"]).to_numpy(dtype=int)
base_rate = float(eligible_label.mean())

def precision_at_k(labels, k):
    return float(np.mean(labels[:min(k, len(labels))]))

p10 = precision_at_k(ranked_label, 10)
p50 = precision_at_k(ranked_label, 50)
print(f"Eligible-set base rate: {base_rate:.3f}")
print(f"Descriptive Precision@10: {p10:.3f} (n={min(10, len(ranked_label))})")
print(f"Descriptive Precision@50: {p50:.3f} (n={min(50, len(ranked_label))})")
print("Top-10 weak pick(s) by this proxy:")
weak = queue.head(10).merge(df[["content_id", "decline_proxy"]], on="content_id", how="left")
weak = weak.loc[weak["decline_proxy"] == 0, ["rank", "content_id", "score"]]
if weak.empty:
    print("None by the descriptive proxy; feature-based failure modes remain possible.")
else:
    display(weak)

# Explicit guard: only the two declared inputs may contribute to score.
score_inputs = {"impressions_90d", "days_since_last_update"}
forbidden = {"trend_direction", "trend_pct", "decline_proxy", "is_declining_label", "content_id", "client_id"}
assert score_inputs.isdisjoint(forbidden)
assert "trend_direction" not in score_inputs and "trend_pct" not in score_inputs
assert queue["rank"].is_monotonic_increasing
assert queue["content_id"].is_unique
print("Leakage check passed: ranking uses only impressions_90d and days_since_last_update.")

metrics = {
    "assignment": "ML-07",
    "dataset": "repo starter CSV; 30,000 anonymized content rows",
    "eligible_rows_impressions_gte_100": int(len(queue)),
    "rule": "log1p(impressions_90d) * min(days_since_last_update, 180) / 180",
    "score_inputs": sorted(score_inputs),
    "reason_code": "visibility_and_staleness",
    "action_label": "editor_review",
    "label_note": "trend_direction == down is a trailing-window descriptive proxy, not a future outcome; never used in score",
    "eligible_decline_proxy_base_rate": round(base_rate, 6),
    "descriptive_precision_at_10": round(p10, 6),
    "descriptive_precision_at_50": round(p50, 6),
    "freshness_bucket_rows": [
        {"bucket": str(r.freshness_bucket), "n": int(r.n), "decline_proxy_rate": None if pd.isna(r.decline_proxy_rate) else round(float(r.decline_proxy_rate), 6)}
        for r in fresh_table.itertuples()
    ],
    "volume_bucket_rows": [
        {"bucket": str(r.volume_bucket), "n": int(r.n), "decline_proxy_rate": None if pd.isna(r.decline_proxy_rate) else round(float(r.decline_proxy_rate), 6)}
        for r in volume_table.itertuples()
    ],
}
METRICS_PATH.write_text(json.dumps(metrics, indent=2) + "\n", encoding="utf-8")
print(f"Wrote run receipt: {METRICS_PATH}")

Eligible-set base rate: 0.598
Descriptive Precision@10: 0.900 (n=10)
Descriptive Precision@50: 0.560 (n=50)
Top-10 weak pick(s) by this proxy:
Leakage check passed: ranking uses only impressions_90d and days_since_last_update.
Wrote run receipt: work/outputs/w04_baseline_metrics.json


 rank           content_id   score
   10 content_cb7e312f5d32 8.35969


## Self-check

- Two signal bucket tables show `n` and cautious, one-word verdicts.
- The scoring rule uses only impressions and update age; it does not use a current trend field or label.
- The queue is ranked and written to `work/outputs/baseline_action_score.csv` (gitignored by the repo on purpose).
- Ten individual rows have an action, reason, and what-could-make-it-wrong note.
- Precision@K is explicitly descriptive because the starter data has no independent future label.
- The executed notebook and small metrics JSON are committed; the queue CSV is regenerated by running this notebook.